In [2]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.classical import expr
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, state_fidelity
from qiskit_aer.noise import NoiseModel, depolarizing_error
import numpy as np


def build_xor(clbits_list, indices):
    if not indices:
        return None
    bits = [clbits_list[i] for i in indices]
    if len(bits) == 1:
        return bits[0]
    result = expr.bit_xor(bits[0], bits[1])
    for b in bits[2:]:
        result = expr.bit_xor(result, b)
    return result


def if_nonzero(qc, condition, gate_fn):
    if isinstance(condition, expr.Expr):
        with qc.if_test(condition):
            gate_fn()
    else:
        with qc.if_test((condition, 1)):
            gate_fn()


def build_noisy_model():
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 1), ['h', 'z', 'x'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(0.02, 2), ['cx', 'cz', 'swap'])
    return noise_model


# ── Dynamic Circuit ────────────────────────────────────────────────────────────

def build_dynamic_circuit_verify(n: int):
    """含 undo Bell state 的版本，用來算 |00⟩ success rate"""
    qc = QuantumCircuit(n, n)
    num_meas = n - 2

    qc.h(0); qc.cx(0, 1)
    for q in range(2, n):  qc.h(q)
    for q in range(1, n-1): qc.cz(q, q+1)
    for i, q in enumerate(range(1, n-1), start=1):
        qc.h(q); qc.measure(q, i-1)

    odd_indices  = list(range(0, num_meas, 2))
    even_indices = list(range(1, num_meas, 2))
    xor_odd  = build_xor(qc.clbits, odd_indices)
    xor_even = build_xor(qc.clbits, even_indices)

    if num_meas % 2 == 1:
        qc.h(n-1)
        if_nonzero(qc, xor_odd, lambda: qc.z(n-1))
        if xor_even: if_nonzero(qc, xor_even, lambda: qc.x(n-1))
    else:
        if xor_even: if_nonzero(qc, xor_even, lambda: qc.x(n-1))
        if_nonzero(qc, xor_odd, lambda: qc.z(n-1))

    # Undo Bell → 驗證用
    qc.cx(0, n-1); qc.h(0)
    print(qc)
    qc.measure(0, num_meas)
    qc.measure(n-1, num_meas+1)
    return qc, num_meas


def build_dynamic_circuit_fidelity(n: int):
    """不含 undo Bell state，correction 完畢後 save density matrix"""
    qc = QuantumCircuit(n, n)
    num_meas = n - 2

    qc.h(0); qc.cx(0, 1)
    for q in range(2, n):  qc.h(q)
    for q in range(1, n-1): qc.cz(q, q+1)
    for i, q in enumerate(range(1, n-1), start=1):
        qc.h(q); qc.measure(q, i-1)

    odd_indices  = list(range(0, num_meas, 2))
    even_indices = list(range(1, num_meas, 2))
    xor_odd  = build_xor(qc.clbits, odd_indices)
    xor_even = build_xor(qc.clbits, even_indices)

    if num_meas % 2 == 1:
        qc.h(n-1)
        if_nonzero(qc, xor_odd, lambda: qc.z(n-1))
        if xor_even: if_nonzero(qc, xor_even, lambda: qc.x(n-1))
    else:
        if xor_even: if_nonzero(qc, xor_even, lambda: qc.x(n-1))
        if_nonzero(qc, xor_odd, lambda: qc.z(n-1))

    # Correction 之後存 density matrix（不做 undo）
    qc.save_density_matrix()
    return qc


# Swap Circuit
def build_swap_circuit_verify(n: int):
    """含 undo Bell state 的版本，用來算 |00⟩ success rate"""
    qc = QuantumCircuit(n, n)
    qc.h(0); qc.cx(0, 1)
    for q in range(1, n-1):
        qc.cx(q, q+1); qc.cx(q+1, q); qc.cx(q, q+1)
    qc.cx(0, n-1); qc.h(0)
    qc.measure(0, n-2)
    qc.measure(n-1, n-1)
    return qc


def build_swap_circuit_fidelity(n: int):
    """不含 undo Bell state，SWAP 完畢後 save density matrix"""
    qc = QuantumCircuit(n, n)
    qc.h(0); qc.cx(0, 1)
    for q in range(1, n-1):
        qc.cx(q, q+1); qc.cx(q+1, q); qc.cx(q, q+1)
    qc.save_density_matrix()
    return qc


# Fidelity 計算 
def compute_bell_fidelity(qc, n, noise_model):
    """
    跑 density matrix simulator，trace out 中間 qubit，
    計算 q0 & q(n-1) 與理想 |Φ+⟩ 的 fidelity
    """
    sim = AerSimulator(method='density_matrix', noise_model=noise_model)
    dm = DensityMatrix(sim.run(transpile(qc, sim), shots=1)
                          .result().data(0)['density_matrix'])

    # little-endian：qubit q 對應 index (n-1-q)
    keep = {n-1-0, n-1-(n-1)}          # q0 和 q(n-1)
    trace_out = [i for i in range(n) if i not in keep]
    rho = partial_trace(dm, trace_out)

    bell_dm = DensityMatrix(Statevector([1, 0, 0, 1] / np.sqrt(2)))
    return state_fidelity(rho, bell_dm)


# ── 主流程 ─────────────────────────────────────────────────────────────────────

def run_sim(n: int = 5):
    noise_model = build_noisy_model()
    noisy_sim   = AerSimulator(noise_model=noise_model)

    # Dynamic 
    dyn_verify_qc, num_meas = build_dynamic_circuit_verify(n)
    dyn_fid_qc              = build_dynamic_circuit_fidelity(n)

    dyn_counts   = noisy_sim.run(transpile(dyn_verify_qc, noisy_sim), shots=4096).result().get_counts()
    dyn_fidelity = compute_bell_fidelity(dyn_fid_qc, n, noise_model)

    dyn_good = sum(v for k, v in dyn_counts.items()
                   if k[-(n-1)] == '0' and k[-n] == '0')

    # Swap 
    swap_verify_qc  = build_swap_circuit_verify(n)
    swap_fid_qc     = build_swap_circuit_fidelity(n)

    swap_counts   = noisy_sim.run(transpile(swap_verify_qc, noisy_sim), shots=4096).result().get_counts()
    swap_fidelity = compute_bell_fidelity(swap_fid_qc, n, noise_model)

    swap_good = sum(v for k, v in swap_counts.items()
                    if k[-(n-1)] == '0' and k[-n] == '0')

    print(f"n={n:2d} | Dynamic : fidelity = {dyn_fidelity:.3f}, |00⟩ success = {dyn_good/4096:.1%}")
    print(f"n={n:2d} | Swap    : fidelity = {swap_fidelity:.3f}, |00⟩ success = {swap_good/4096:.1%}")
    print()


if __name__ == "__main__":
    for n in range(3, 7):
        run_sim(n)

     ┌───┐                                               ┌───┐
q_0: ┤ H ├──■─────────────────────────────────────────■──┤ H ├
     └───┘┌─┴─┐   ┌───┐┌─┐                            │  └───┘
q_1: ─────┤ X ├─■─┤ H ├┤M├────────────────────────────┼───────
     ┌───┐└───┘ │ ├───┤└╥┘  ┌──────  ┌───┐ ───────┐ ┌─┴─┐     
q_2: ┤ H ├──────■─┤ H ├─╫───┤ If-0  ─┤ Z ├  End-0 ├─┤ X ├─────
     └───┘        └───┘ ║   └──╥───  └───┘ ───────┘ └───┘     
                        ║ ┌────╨────┐                         
c: 3/═══════════════════╩═╡ c_0=0x1 ╞═════════════════════════
                        0 └─────────┘                         
n= 3 | Dynamic : fidelity = 0.949, |00⟩ success = 93.7%
n= 3 | Swap    : fidelity = 0.937, |00⟩ success = 91.6%

     ┌───┐                                                                  »
q_0: ┤ H ├──■───────────────────────────────────────────────────────────────»
     └───┘┌─┴─┐   ┌───┐┌─┐                                                  »
q_1: ─────┤ X ├─■─┤ H ├